# Smoke test: Indic-Speak vs Orpheus-Hindi on MaSaC

5 code-switched MaSaC utterances → each model, to check that both load, produce audio, and handle Latin-script English inside Hindi.

- **Indic-Speak** (`bodhan-ai/indic-speak`): Hindi voices Kavya (F) and Amit (M); vendor decoding config (t 0.6, top_p 0.9, rep. penalty 1.2).
- **Orpheus 3B Hindi** (`canopylabs/3b-hi-ft-research_release`): voice name undocumented, so all 5 utterances with no voice prefix, plus a voice-name sweep on one utterance. Decoding t 0.6, top_p 0.9, rep. penalty 1.1.

Kernel: `/local/speech/users/sp4050/envs/tts/bin/python`. Wavs are also written to `/local/speech/users/sp4050/outputs/smoke/`.

⚠️ Executed copies of this notebook embed MaSaC text and audio — clear outputs before committing (the repo is public).

In [ ]:
import os
os.environ.setdefault("HF_HOME", "/local/speech/users/sp4050/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # weights are already cached
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")    # check nvidia-smi first; GPUs are shared

import gc, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, Markdown, display

LOCAL = Path("/local/speech/users/sp4050")
OUT = LOCAL / "outputs/smoke"
SR = 24_000
DEVICE = "cuda"
SEED = 1234

def cached_snapshot(repo):
    """Local snapshot dir of a cached HF repo (we skipped some unneeded files, so snapshot_download() calls it incomplete)."""
    d = Path(os.environ["HF_HOME"]) / "hub" / ("models--" + repo.replace("/", "--"))
    return d / "snapshots" / (d / "refs/main").read_text().strip()

INDIC_DIR = cached_snapshot("bodhan-ai/indic-speak")
ORPHEUS_DIR = cached_snapshot("canopylabs/3b-hi-ft-research_release")
print(torch.__version__, torch.cuda.get_device_name(0))

## 5 code-switched utterances (hand-picked)
Much of MSH-COMICS `Utterances.txt` is garbled (machine-transliteration-like noise, scrambled word order, duplicate utterances in different forms), so random sampling picks unusable text. For the smoke test we use 5 hand-checked clean lines (0-indexed line numbers) that mix Devanagari Hindi with Latin-script English.

In [ ]:
UTT_FILE = LOCAL / "data/masac/MSH-COMICS/Data/Raw_Utterances/Utterances.txt"
LINES = [16, 44, 48, 126, 270]

all_lines = UTT_FILE.read_text(encoding="utf-8").splitlines()
utts = []
for i in LINES:
    speaker, _, text = all_lines[i].partition(":")
    utts.append((i, speaker.strip(), text.strip()))
    print(f"[{i}] {speaker.strip()}: {text.strip()}")

## Helpers

In [ ]:
results = []

def run(model, generate, voice, items=None, **kw):
    """Synthesize each utterance, save wav, show an audio player, log stats."""
    out_dir = OUT / model / (voice or "no_voice")
    out_dir.mkdir(parents=True, exist_ok=True)
    display(Markdown(f"### {model} · voice: `{voice or '(none)'}`"))
    for i, speaker, text in items or utts:
        try:
            wav, info = generate(text, voice, **kw)
        except Exception as e:
            print(f"[{i}] FAILED: {e}")
            results.append(dict(model=model, voice=voice, utt=i, error=str(e)))
            continue
        sf.write(out_dir / f"{i}.wav", wav, SR)
        results.append(dict(model=model, voice=voice, utt=i, **info))
        flag = " · ⚠️ hit max_new_tokens" if info["hit_max"] else ""
        display(Markdown(f"**[{i}] {speaker}:** {text}  \n"
                         f"`{info['audio_s']:.1f}s audio · {info['gen_s']:.1f}s gen · RTF {info['rtf']:.2f} · {info['n_tokens']} tokens{flag}`"))
        display(Audio(wav, rate=SR))

def token_budget(text):
    # ~82 audio tokens/s, ~12 chars/s speech → 2x headroom, min 4 s
    return int(82 * max(4, len(text) / 12) * 2)

def free(*names):
    for n in names:
        globals().pop(n, None)
    gc.collect(); torch.cuda.empty_cache()

## Indic-Speak
Reuses the repo's own `build_prompt` / `ids_to_codes` / decoders, but calls `generate` directly so we can pass `repetition_penalty=1.2` (the vendor's benchmark config; their `inference.py` omits it).

`vocos/best.pt` is a pickle and the repo's loader opens it with `weights_only=False`. We first confirm it loads with `weights_only=True` — if that succeeds, the file contains only plain tensors/containers and can't execute code.

In [ ]:
sys.path.insert(0, str(INDIC_DIR))
import inference as indic

torch.load(INDIC_DIR / "vocos/best.pt", map_location="cpu", weights_only=True)  # raises if unsafe content
print("vocos checkpoint passes weights_only=True")

t0 = time.time()
tts = indic.TTS(str(INDIC_DIR), device=DEVICE)
print(f"Indic-Speak loaded in {time.time() - t0:.0f}s · {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
def indic_generate(text, voice, style="", temperature=0.6, top_p=0.9, top_k=50, repetition_penalty=1.2, seed=SEED):
    torch.manual_seed(seed)
    prompt = indic.build_prompt(tts.tok, text, voice, style)
    max_new = token_budget(text)
    ids = torch.tensor([prompt], device=DEVICE)
    t0 = time.time()
    with torch.no_grad():
        gen = tts.lm.generate(
            input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new,
            eos_token_id=[tts._eos, tts.tok.eos_token_id], pad_token_id=tts.tok.eos_token_id,
            do_sample=True, temperature=temperature, top_p=top_p, top_k=top_k,
            repetition_penalty=repetition_penalty)
        new = gen[0, len(prompt):].tolist()
        codes = indic.ids_to_codes(new, tts._base, DEVICE)
        wav = tts.vocos(tts.snac.quantizer.from_codes(codes).float())[0, 0].clamp(-1, 1).float().cpu().numpy()
    gen_s, audio_s = time.time() - t0, len(wav) / SR
    return wav, dict(n_tokens=len(new), hit_max=len(new) >= max_new, gen_s=gen_s, audio_s=audio_s, rtf=gen_s / audio_s)

In [ ]:
run("indic-speak", indic_generate, "Kavya")

In [ ]:
run("indic-speak", indic_generate, "Amit")

In [ ]:
free("tts")

## Orpheus 3B Hindi
Standard Orpheus prompt: `<custom_token_3>` + tokenized `"{voice}: {text}"` + `<|eot_id|>` + `<custom_token_4>`; audio follows `<custom_token_1>` and ends at `<custom_token_2>`. Audio tokens start at id 128266 with the same 7-tokens-per-frame SNAC layout as Indic-Speak, so `ids_to_codes` is reused; decoding uses the stock SNAC 24 kHz decoder.

In [ ]:
from snac import SNAC
from transformers import AutoModelForCausalLM, AutoTokenizer

START_HUMAN, END_TEXT, END_HUMAN = 128259, 128009, 128260
START_SPEECH, END_SPEECH, AUDIO_BASE = 128257, 128258, 128266

t0 = time.time()
o_tok = AutoTokenizer.from_pretrained(ORPHEUS_DIR)
o_lm = AutoModelForCausalLM.from_pretrained(ORPHEUS_DIR, dtype=torch.bfloat16, attn_implementation="sdpa").to(DEVICE).eval()
snac24 = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to(DEVICE).eval()
print(f"Orpheus loaded in {time.time() - t0:.0f}s · {torch.cuda.memory_allocated() / 1e9:.1f} GB")

def orpheus_generate(text, voice, temperature=0.6, top_p=0.9, repetition_penalty=1.1, seed=SEED):
    torch.manual_seed(seed)
    prompt = [START_HUMAN] + o_tok(f"{voice}: {text}" if voice else text).input_ids + [END_TEXT, END_HUMAN]
    max_new = token_budget(text)
    ids = torch.tensor([prompt], device=DEVICE)
    t0 = time.time()
    with torch.no_grad():
        gen = o_lm.generate(
            input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new,
            eos_token_id=END_SPEECH, pad_token_id=END_SPEECH,
            do_sample=True, temperature=temperature, top_p=top_p, repetition_penalty=repetition_penalty)
        new = gen[0, len(prompt):].tolist()
        n_new = len(new)
        if START_SPEECH in new:  # keep tokens after the last start-of-speech marker
            new = new[len(new) - new[::-1].index(START_SPEECH):]
        codes = indic.ids_to_codes(new, AUDIO_BASE, DEVICE)
        wav = snac24.decode(codes)[0, 0].clamp(-1, 1).float().cpu().numpy()
    gen_s, audio_s = time.time() - t0, len(wav) / SR
    return wav, dict(n_tokens=n_new, hit_max=n_new >= max_new, gen_s=gen_s, audio_s=audio_s, rtf=gen_s / audio_s)

In [ ]:
run("orpheus-hi-ft", orpheus_generate, None)

### Voice-name sweep (first utterance only)
Candidate names are guesses — the Hindi voice is not documented. Listen for which (if any) gives a stable, consistent voice vs. the no-voice baseline above.

In [ ]:
for voice in ["ऋतिका", "ritika", "tara"]:
    run("orpheus-hi-ft", orpheus_generate, voice, items=utts[:1])

In [ ]:
free("o_lm", "snac24")

## Summary

In [ ]:
df = pd.DataFrame(results)
pd.set_option("display.precision", 2)
df

In [ ]:
df.groupby(["model", "voice"], dropna=False)[["audio_s", "gen_s", "rtf", "hit_max"]].mean()